# Ep. 01 — El mapa de AI Engineering

Curso 1 — Fundamentos de AI Engineering · Linear [CM-41](https://linear.app/caiomedeiros/issue/CM-41)


## Objetivos de aprendizaje

Al final de este notebook podrás:
1. Distinguir **AI Engineer**, **Prompt Engineer** y **ML Engineer**.
2. Dibujar el mapa del canal: **datos → modelo → sistema → producto → operaciones**.
3. Decidir en qué capa está el problema antes de escribir un prompt.


## Gancho — tres roles, una confusión

La gente mezcla *prompt engineering*, *ML engineering* y *AI engineering*. Esa confusión pierde tiempo y entrega demos frágiles. Este episodio corrige el vocabulario y entrega un mapa práctico.


## Definición precisa

**AI Engineering** es la disciplina de convertir capacidad del modelo en **comportamiento fiable de producto**: contratos, retrieval, tools, guardrails, evaluación, coste/latencia y operaciones — no solo prompts ingeniosos.

| Rol | Enfoque principal | Artefacto típico |
|-----|-------------------|------------------|
| Prompt Engineer | Redacción y few-shots | Plantillas de prompt |
| ML Engineer | Entrenamiento / fine-tune / serving | Modelos y pipelines |
| AI Engineer | Sistema alrededor del modelo | Features de IA en producción |


## Mapa: datos → modelo → sistema → producto → operaciones

Toda feature de IA atraviesa estas capas. Saltar una capa es cómo mueren las demos en producción.

Ejecuta la celda para imprimir el mapa y puntuar un ticket ficticio.


In [ ]:
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass, field
from typing import Any, Callable

# Optional: set OPENAI_API_KEY to call a real OpenAI-compatible API.
# Without a key, every demo below still runs offline with a tiny mock LLM.
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
API_BASE = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1").rstrip("/")
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")


def chat(messages: list[dict[str, str]], *, temperature: float = 0.2, max_tokens: int = 300) -> str:
    """Vanilla chat completion: direct HTTP to an OpenAI-compatible API, or offline mock."""
    if not API_KEY:
        return _mock_chat(messages)
    try:
        import urllib.request

        payload = {
            "model": MODEL,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        req = urllib.request.Request(
            f"{API_BASE}/chat/completions",
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        return data["choices"][0]["message"]["content"]
    except Exception as exc:  # noqa: BLE001 — educational fallback
        return f"[API error → offline mock] {exc}\n\n" + _mock_chat(messages)


def _mock_chat(messages: list[dict[str, str]]) -> str:
    """Deterministic offline stand-in so the lesson runs without credentials."""
    user = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
    system = next((m["content"] for m in messages if m["role"] == "system"), "")
    lower = user.lower()

    if "json" in system.lower() or "json" in lower:
        return '{"ok": true, "summary": "mock structured answer", "confidence": 0.61}'
    if "capital of france" in lower or "capital de frança" in lower or "capital de francia" in lower:
        return "Paris"
    if "2+2" in lower or "2 + 2" in lower:
        return "4"
    if "invent" in lower or "invente" in lower or "inventa" in lower:
        return "The 1847 Treaty of New Avalon was signed by Admiral Kestrel."
    if "email" in lower or "ssn" in lower or "cpf" in lower:
        return "Sure — contact jane.doe@example.com / SSN 123-45-6789."
    # Default: echo a short, slightly overconfident reply
    snippet = user.strip().replace("\n", " ")[:120]
    return f"Based on my training, the answer is clearly related to: {snippet}…"


print("Mode:", "LIVE API" if API_KEY else "OFFLINE MOCK (set OPENAI_API_KEY for live calls)")
print("Model:", MODEL if API_KEY else "mock-llm")


In [ ]:
LAYERS = [
    ("data", "sources, cleaning, labels, permissions"),
    ("model", "provider choice, prompts, decoding, fine-tunes"),
    ("system", "retrieval, tools, memory, guardrails, orchestration"),
    ("product", "UX contracts, latency SLO, fallbacks, pricing"),
    ("operations", "eval, monitoring, cost, incident response"),
]

ROLE_FOCUS = {
    "prompt_engineer": {"model"},
    "ml_engineer": {"data", "model"},
    "ai_engineer": {"data", "model", "system", "product", "operations"},
}

def print_map() -> None:
    print("AI Engineering map")
    print("=" * 40)
    for name, blurb in LAYERS:
        print(f"→ {name:12} {blurb}")

def score_ticket(description: str, suspected_layer: str) -> dict[str, Any]:
    keywords = {
        "data": ["dataset", "label", "pii", "schema", "csv", "permission"],
        "model": ["prompt", "temperature", "hallucin", "token", "fine-tune"],
        "system": ["retriev", "rag", "tool", "agent", "guardrail", "memory"],
        "product": ["button", "ux", "latency", "sla", "fallback", "pricing"],
        "operations": ["eval", "metric", "monitor", "cost", "incident", "alert"],
    }
    text = description.lower()
    hits = {layer: sum(1 for k in keys if k in text) for layer, keys in keywords.items()}
    guessed = max(hits, key=hits.get)
    return {
        "suspected_layer": suspected_layer,
        "suggested_layer": guessed,
        "keyword_hits": hits,
        "aligned": suspected_layer == guessed,
    }

print_map()
print()
ticket = "Chat answers cite the wrong PDF; retrieval returns unrelated chunks under load."
result = score_ticket(ticket, suspected_layer="system")
print("Ticket:", ticket)
print(json.dumps(result, indent=2))
assert result["suggested_layer"] == "system"
print("\nChecklist: name the broken layer before editing the prompt.")


## Checklist de acción

Antes de abrir el chat, responde: **¿qué capa está rota?** Si no puedes nombrar la capa, estás adivinando.


## Qué cubre este canal (y qué no)

- **Cubre (Fundamentos):** tokens/contexto, prompt vs sistema, eval básica, riesgos.
- **Pilares siguientes:** sistemas más profundos, producción, evals/frameworks avanzados.
- **No está aquí:** entrenar foundation models desde cero.


## Takeaways

1. Habilidad de prompt ≠ AI Engineering.
2. El mapa te obliga a localizar el fallo antes de cambiar palabras.
3. Siguiente episodio: tokens, ventana de contexto y por qué el modelo “falla” sin ser “tonto”.
